# 🎭 GANs Part B: Training Challenges, Sampling & Evaluation

## Training Challenges

Training a GAN is famously unstable. Three core failure modes:

---

### 1. Vanishing Gradients
**What happens:** The discriminator becomes too good too fast. It confidently rejects all fakes, outputting near-zero probability. The generator receives near-zero gradients — it can't learn from a signal that says nothing useful.

**Analogy:** A student takes an exam, gets 0/100, and the grader only writes "wrong" with no further feedback. The student can't improve.

**Mitigation:** Use Wasserstein loss (see below), add noise to discriminator inputs, or use label smoothing.

---

### 2. Mode Collapse
**What happens:** The generator finds one (or a few) outputs that consistently fool the discriminator and gets stuck producing only those outputs. It ignores entire regions of the data distribution.

**Analogy:** A chef learns that the judges always love cheesecake, so they serve cheesecake every single round — completely forgetting how to cook anything else.

**Mitigation:** Mini-batch discrimination, unrolled GANs, Wasserstein loss, or diversity-promoting regularization.

---

### 3. Failure to Converge
**What happens:** Generator and discriminator oscillate — each improving only to destabilize the other. Loss curves show no steady progress; the system never reaches Nash equilibrium.

**Analogy:** Two negotiators who keep escalating demands in response to each other, never agreeing, just cycling forever.

**Mitigation:** Careful learning rate tuning, more discriminator steps per generator step, spectral normalization.

---

### Wasserstein Loss (WGAN) — The Core Fix
Instead of training the discriminator as a classifier outputting probabilities, WGAN trains a **critic** that outputs an unbounded real score. The loss is the **Wasserstein distance** (Earth Mover's distance) between real and fake distributions.

| | Vanilla GAN | WGAN |
|---|---|---|
| Discriminator output | Probability [0, 1] | Unbounded score |
| Loss when D is confident | ~0 (vanishes) | Still informative |
| Training stability | Low | High |
| Constraint | None | Lipschitz (weight clipping or GP) |

**Key insight:** Even when the real and fake distributions have no overlap, Wasserstein distance still gives a meaningful gradient signal proportional to how far apart the distributions are.

In [ ]:
import torch
import torch.nn.functional as F

# --- Vanilla GAN losses (BCE) ---
def vanilla_discriminator_loss(real_scores, fake_scores):
    real_loss = F.binary_cross_entropy_with_logits(real_scores, torch.ones_like(real_scores))
    fake_loss = F.binary_cross_entropy_with_logits(fake_scores, torch.zeros_like(fake_scores))
    return real_loss + fake_loss

def vanilla_generator_loss(fake_scores):
    return F.binary_cross_entropy_with_logits(fake_scores, torch.ones_like(fake_scores))

# --- Wasserstein (WGAN) losses ---
def wasserstein_critic_loss(real_scores, fake_scores):
    return fake_scores.mean() - real_scores.mean()  # minimize: push real up, fake down

def wasserstein_generator_loss(fake_scores):
    return -fake_scores.mean()  # maximize critic score on fakes

# --- Compare gradient magnitude when discriminator is very confident ---
real = torch.tensor([10.0])   # discriminator is very confident real = real
fake = torch.tensor([-10.0])  # discriminator is very confident fake = fake

fake_req = fake.clone().requires_grad_(True)
v_loss = vanilla_generator_loss(fake_req)
v_loss.backward()
vanilla_grad = fake_req.grad.item()

fake_req2 = fake.clone().requires_grad_(True)
w_loss = wasserstein_generator_loss(fake_req2)
w_loss.backward()
wgan_grad = fake_req2.grad.item()

print(f"Generator gradient (Vanilla GAN, confident D): {vanilla_grad:.6f}  <- nearly zero!")
print(f"Generator gradient (WGAN, confident D):        {wgan_grad:.6f}  <- constant, informative")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n = 500

# Healthy generator: samples spread across multiple modes
healthy = np.concatenate([
    np.random.randn(n // 2, 2) + np.array([2, 2]),
    np.random.randn(n // 2, 2) + np.array([-2, -2])
])

# Collapsed generator: all samples near one point
collapsed = np.random.randn(n, 2) * 0.1 + np.array([2, 2])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.scatter(*collapsed.T, alpha=0.4, s=15, color='tomato')
ax1.set_title('Mode Collapse\n(all outputs near one point)', fontsize=12)
ax1.set_xlim(-5, 5); ax1.set_ylim(-5, 5)
ax2.scatter(*healthy.T, alpha=0.4, s=15, color='steelblue')
ax2.set_title('Healthy Generator\n(covers multiple modes)', fontsize=12)
ax2.set_xlim(-5, 5); ax2.set_ylim(-5, 5)
for ax in (ax1, ax2):
    ax.set_aspect('equal'); ax.axhline(0, lw=0.5, color='gray'); ax.axvline(0, lw=0.5, color='gray')
plt.tight_layout()
plt.show()

## Latent Space & Sampling Strategies

The **latent vector z** is the generator's input — a point in a high-dimensional noise space (e.g., 128-D Gaussian). The generator learns a mapping from this space to the image space.

### Random Sampling
Draw z ~ N(0, I). Full coverage of the latent space. Occasionally samples from low-density regions where the generator hasn't learned well — produces artifacts or odd images.

### Truncated Sampling
Only accept z where ||z|| < threshold (e.g., 0.7). Samples stay near the center of the distribution where the generator is best trained.

**Quality-Diversity Tradeoff:**

| Truncation threshold | Quality | Diversity |
|---|---|---|
| Low (e.g., 0.5) | High (sharp, clean) | Low (repetitive) |
| High (e.g., 2.0) | Lower (more artifacts) | High (varied) |
| No truncation | Lowest near tails | Highest |

**Used in practice:** BigGAN and StyleGAN both use truncation at inference time to control the quality-diversity tradeoff. The truncation trick is applied **only at inference**, not during training.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
threshold = 0.7
n_samples = 2000

# Full N(0,1) sampling
full_samples = np.random.randn(n_samples)

# Truncated: resample until |z| < threshold
truncated = []
while len(truncated) < n_samples:
    z = np.random.randn(n_samples)
    truncated.extend(z[np.abs(z) < threshold])
truncated = np.array(truncated[:n_samples])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.hist(full_samples, bins=50, color='steelblue', edgecolor='white')
ax1.set_title('Full N(0,1) Sampling\n(high diversity, possible artifacts)', fontsize=11)
ax1.set_xlim(-4, 4)
ax2.hist(truncated, bins=50, color='darkorange', edgecolor='white')
ax2.axvline(-threshold, color='red', linestyle='--', label=f'threshold ±{threshold}')
ax2.axvline(threshold, color='red', linestyle='--')
ax2.set_title(f'Truncated Sampling (|z| < {threshold})\n(high quality, lower diversity)', fontsize=11)
ax2.set_xlim(-4, 4); ax2.legend()
plt.tight_layout()
plt.show()

## Evaluation: FID and Inception Score

### Inception Score (IS)
Run generated images through a pretrained Inception network. A good generator should:
1. Produce images that look clearly like *something* (low conditional entropy: p(y|x) is peaked)
2. Generate *diverse* outputs across many classes (high marginal entropy: p(y) is flat)

IS = exp(E[KL(p(y|x) || p(y))]). Higher is better. **Limitation:** Does not measure alignment with real data distribution.

---

### Frechet Inception Distance (FID) — The Gold Standard
Compares the distribution of real images vs generated images in Inception feature space.

**Step-by-step:**
1. Extract Inception v3 pool3 features (2048-D) for ~10k real images → compute mean `mu_r` and covariance `Sigma_r`
2. Do the same for ~10k generated images → `mu_g`, `Sigma_g`
3. Compute: `FID = ||mu_r - mu_g||^2 + Tr(Sigma_r + Sigma_g - 2*sqrt(Sigma_r * Sigma_g))`

Lower FID = distributions are more similar = better generator.

---

### Interview Cheat Sheet

**Training Challenges:**
- Vanishing gradients -> discriminator too good -> use WGAN
- Mode collapse -> generator ignores modes -> mini-batch discrimination, WGAN
- No convergence -> oscillation -> spectral norm, careful LR, more D steps

**WGAN key facts:**
- Critic (not discriminator) outputs unbounded scores
- Loss = E[critic(fake)] - E[critic(real)] for critic; -E[critic(fake)] for generator
- Requires Lipschitz constraint: weight clipping (original) or gradient penalty (WGAN-GP, preferred)

**Evaluation key facts:**
- FID needs ~10k samples for stable estimates; lower is better
- IS measures quality + diversity but ignores real data distribution
- FID is preferred in practice; IS can be fooled by diverse-but-unrealistic images
- Truncation trick: used at inference only; trades diversity for quality

## Quiz

**Q1: Why does Wasserstein loss fix the vanishing gradient problem?**

<details>
<summary>Show Answer</summary>

The WGAN critic outputs unbounded real-valued scores rather than probabilities. Even when the critic perfectly separates real from fake (e.g., assigning +100 to reals and -100 to fakes), the gradient of the Wasserstein loss with respect to the generator is still constant and non-zero (-1 per sample). In contrast, vanilla GAN's sigmoid output saturates near 0 or 1, making the gradient of BCE loss approach zero when the discriminator is confident — starving the generator of learning signal.

</details>

---

**Q2: A GAN produces very sharp, photorealistic faces but they all look nearly identical. What is happening and how would you detect it?**

<details>
<summary>Show Answer</summary>

This is mode collapse. The generator has found a small region of image space that fools the discriminator and stopped exploring. You can detect it by: (1) computing FID — it will be high because the generated distribution doesn't match the real distribution's spread; (2) measuring perceptual diversity (LPIPS distance between pairs of generated images — collapsed output will show very low pairwise distances); (3) visually inspecting a grid of samples.

</details>

---

**Q3: You reduce the truncation threshold from 1.5 to 0.5 in a trained BigGAN. What tradeoff are you making, and why does this only work at inference time?**

<details>
<summary>Show Answer</summary>

Reducing the threshold increases image quality (sharper, more realistic) but decreases diversity (fewer distinct outputs, closer to the "average" of each class). This works only at inference because the model was trained with full N(0,1) latent vectors — the generator learned to handle the full distribution. Applying truncation at inference simply restricts sampling to the high-density region where the generator is best calibrated. If you trained with truncated inputs, the generator would never learn to handle the tails and would still produce poor outputs there.

</details>